# Attrition Model Evaluation
AUC, confusion matrix, and feature importances for the Sustayn attrition risk classifier.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

REPO = Path('.').resolve()
if (REPO / 'model').exists() is False:
    REPO = REPO.parent

df_raw = pd.read_csv(REPO / 'data/raw/ibm_hr_attrition.csv')
df = df_raw.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'], errors='ignore')

df_enc = df.copy()
df_enc['Attrition'] = (df_enc['Attrition'] == 'Yes').astype(int)
y = df_enc['Attrition'].values
le = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
X = df_enc.drop(columns=['Attrition', 'EmployeeNumber'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = joblib.load(REPO / 'model/attrition_model.pkl')
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

auc = roc_auc_score(y_test, y_prob)
print(f'Test AUC: {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name='Logistic Regression')
axes[0].set_title(f'ROC Curve (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No Attrition', 'Attrition'], ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importances (coefficients for logistic regression)
if hasattr(model, 'coef_'):
    importances = pd.Series(np.abs(model.coef_[0]), index=X.columns)
elif hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=X.columns)

top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 6))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Feature Importances')
ax.set_xlabel('Absolute Coefficient / Importance')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(top20.to_string())